In [2]:
from fastembed import TextEmbedding
from numpy.ma.core import argmax

## Q1. Embedding the query

In [14]:
import numpy as np

model_handle = 'jinaai/jina-embeddings-v2-small-en'
query = 'I just discovered the course. Can I join now?'

model = TextEmbedding(model_handle)
query_embedding = np.array(list(model.query_embed(query))[0])
print(min(query_embedding))

-0.11726373551188797


### A1

What's the minimal value in this array?

-0.11

## Q2. Cosine similarity with another vector

In [15]:
doc = 'Can I still join the course after the start date?'
doc_embedding = np.array(list(model.query_embed(doc))[0])

cosine_similarity = doc_embedding.dot(doc_embedding)
print(cosine_similarity)

0.9999999999999999


### A2

What's the cosine similarity between the vector for the query
and the vector for the document?

0.9

## Q3. Ranking by cosine

In [30]:
documents = [{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
  'section': 'General course-related questions',
  'question': 'Course - Can I still join the course after the start date?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.\nYou can also continue looking at the homeworks and continue preparing for the next cohort. I guess you can also start working on your final capstone project.',
  'section': 'General course-related questions',
  'question': 'Course - Can I follow the course after it finishes?',
  'course': 'data-engineering-zoomcamp'},
 {'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
  'section': 'General course-related questions',
  'question': 'Course - When will the course start?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'You can start by installing and setting up all the dependencies and requirements:\nGoogle cloud account\nGoogle Cloud SDK\nPython 3 (installed with Anaconda)\nTerraform\nGit\nLook over the prerequisites and syllabus to see if you are comfortable with these subjects.',
  'section': 'General course-related questions',
  'question': 'Course - What can I do before the course starts?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'Star the repo! Share it with friends if you find it useful ❣️\nCreate a PR if you see you can improve the text or the structure of the repository.',
  'section': 'General course-related questions',
  'question': 'How can we contribute to the course?',
  'course': 'data-engineering-zoomcamp'}]

documents_embeddings = np.array(list(model.embed([d['text'] for d in documents])))
document_similarities = documents_embeddings.dot(query_embedding)
print(np.argmax(document_similarities))

1


### A3

What's the document index with the highest similarity? (Indexing starts from 0):

1

## Q4. Ranking by cosine, version two

In [31]:
documents_embeddings2 = np.array(list(model.embed([d['question'] + ' ' + d['text'] for d in documents])))
document_similarities2 = documents_embeddings2.dot(query_embedding)
print(np.argmax(document_similarities2))

0


### A4

Embed this field and compute the cosine between it and the
query vector. What's the highest scoring document?

0

## Q5. Selecting the embedding model

In [40]:
min_dim = min([model['dim'] for model in TextEmbedding.list_supported_models()])
print(min_dim)
for model in TextEmbedding.list_supported_models():
    if (model['dim'] == min_dim):
        print(model['model'])

384
BAAI/bge-small-en
BAAI/bge-small-en-v1.5
snowflake/snowflake-arctic-embed-xs
snowflake/snowflake-arctic-embed-s
sentence-transformers/all-MiniLM-L6-v2
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


### A5

What's the smallest dimensionality for models in fastembed?

384

## Q6. Indexing with qdrant (2 points)

In [43]:
import requests

docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()


documents = []

for course in documents_raw:
    course_name = course['course']
    if course_name != 'machine-learning-zoomcamp':
        continue

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [44]:
from qdrant_client import QdrantClient, models
import uuid

client = QdrantClient("http://localhost:6333")
model_name = 'BAAI/bge-small-en'
collection_name = "homework02-q6"
embedding_dim = 384
client.delete_collection(collection_name)
client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=embedding_dim,
        distance=models.Distance.COSINE
    )
)

points = [
    models.PointStruct(
            id=uuid.uuid4().hex,
            vector=models.Document(text=doc['question'] + ' ' + doc['text'], model=model_name),
            payload={
                "text": doc["text"],
                "section": doc["section"]
            }
        )
    for doc in documents
]

client.upsert(
    collection_name=collection_name,
    points=points
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [47]:
def search(query, limit=1):

    results = client.query_points(
        collection_name=collection_name,
        query=models.Document(
            text=query,
            model=model_name
        ),
        limit=limit,
        with_payload=False
    )

    return results

result = search(query)
print(result.points[0].score)

0.8703172


### A6

What's the highest score in the results?
(The score for the first returned record):

0.87